# BRCA1 ClinProtGym data overview

A minimal, read-only tour of how the `MV_BRCA1_Findlay_2018` data is stored. Every cell just loads a piece of the dataset and prints it as a DataFrame (or reports array shapes). Nothing here submits jobs, trains, or writes files.

The data has four logical layers:

| Layer | What it is | Where it lives |
|-------|-----------|----------------|
| **Sequences** | The BRCA1 reference protein + one full-length mutated protein per variant | `processed_state.pkl` (`reference_sequence`, `sequence_to_protein_sequence`), `adapter_counts.csv` (`mutated_sequence`) |
| **Mutations** | Parsed variant identity (`wt_aa`,`position`,`mutant_aa`) + ClinVar clinical labels | `sequence_metadata`, `annotations_dataframe`, `sequence_to_mutation_sites` |
| **Count data** | Raw DMS read counts across conditions, popDMS trajectory table, functional scores | `adapter_counts.csv`, `sequence_dataframe`, `scores_dataframe` |
| **Protein embeddings** | Per-layer max-pooled ESM-C vectors, popDMS inference results, learned SAE features | `*_Layer_{L}_seq_to_features.pkl`, `*_inference_results.pkl`, SAE `feature_path` pickles |

The join key everywhere is **`SequenceIndex`** (equivalently `mutant`), an HGVS-style protein string like `I1855M` = wild-type `I` at position `1855` mutated to `M`.

In [1]:
from pathlib import Path
import pickle
import glob

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)

# Locate the repo root and the BRCA1 dataset directory.
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "esmDMS.py").exists():
    for parent in Path.cwd().parents:
        if (parent / "esmDMS.py").exists():
            REPO_ROOT = parent
            break

DATASET = "MV_BRCA1_Findlay_2018"
DATASET_ROOT = REPO_ROOT / "data" / "clinprotgym_esmc_sae" / "datasets" / DATASET
SEQ_DIR = DATASET_ROOT / "sequence_data"
TABLE_DIR = DATASET_ROOT / "tables"

print("repo root: ", REPO_ROOT)
print("dataset dir:", DATASET_ROOT)
assert DATASET_ROOT.is_dir(), "BRCA1 dataset directory not found"

repo root:  /net/dali/home/barton/dhw28/popDMS/esmDMS
dataset dir: /net/dali/home/barton/dhw28/popDMS/esmDMS/data/clinprotgym_esmc_sae/datasets/MV_BRCA1_Findlay_2018


## 0. The processed-state pickle

Almost everything except the embeddings is bundled into a single pickle. Load it once and reuse.

In [2]:
state_path = SEQ_DIR / f"{DATASET}_processed_state.pkl"
with state_path.open("rb") as handle:
    state = pickle.load(handle)

print(f"processed_state.pkl top-level keys ({len(state)}):\n")
for key, value in state.items():
    if isinstance(value, pd.DataFrame):
        detail = f"DataFrame  shape={value.shape}"
    elif isinstance(value, dict):
        detail = f"dict       n={len(value)}"
    elif isinstance(value, str):
        detail = f"str        len={len(value)}  {value[:50]!r}"
    else:
        detail = f"{type(value).__name__}"
    print(f"  {key:32s} {detail}")

processed_state.pkl top-level keys (20):

  dataset                          str        len=21  'MV_BRCA1_Findlay_2018'
  raw_csv                          str        len=76  '/net/dali/home/barton/dhw28/popDMS/esmDMS/new_data'
  raw_csv_size                     int
  raw_csv_mtime_ns                 int
  sequence_transform               str        len=4  'full'
  context_length                   NoneType
  window_stride                    NoneType
  ingestion_version                int
  paths                            dict       n=19
  reference_sequence               str        len=1863  'MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLK'
  full_reference_sequence          str        len=1863  'MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLK'
  reference_kind                   str        len=7  'protein'
  sequence_dataframe               DataFrame  shape=(11022, 7)
  sequence_to_protein_sequence     dict       n=1838
  sequence_to_mutation_sites       dict       n=1838
  seque

## 1. Sequences

The wild-type BRCA1 protein is 1863 aa. Each variant has a full-length mutated protein that differs from wild-type at exactly the mutation site(s).

In [3]:
ref = state["reference_sequence"]
print(f"reference (wild-type) BRCA1 protein: length={len(ref)} aa")
print(ref[:80] + " ...\n")

seq_map = state["sequence_to_protein_sequence"]  # mutant -> full protein
print(f"sequence_to_protein_sequence: {len(seq_map)} entries (includes '__wildtype__')")

# Show a few variants next to the wild-type residue they change.
rows = []
for mutant, protein in list(seq_map.items()):
    diffs = [(i, ref[i], protein[i]) for i in range(len(ref)) if ref[i] != protein[i]]
    rows.append({
        "mutant": mutant,
        "protein_length": len(protein),
        "n_diffs_vs_wt": len(diffs),
        "change (0-based pos: wt->mut)": "; ".join(f"{p}: {w}->{m}" for p, w, m in diffs),
        "protein_sequence": protein,
    })
display(pd.DataFrame(rows))

reference (wild-type) BRCA1 protein: length=1863 aa
MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFS ...

sequence_to_protein_sequence: 1838 entries (includes '__wildtype__')


,mutant,protein_length,n_diffs_vs_wt,change (0-based pos: wt->mut),protein_sequence
0,__wildtype__,1863,0,,MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKF...
1,I1855M,1863,1,1854: I->M,MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKF...
2,I1855R,1863,1,1854: I->R,MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKF...
3,I1855T,1863,1,1854: I->T,MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKF...
4,I1855K,1863,1,1854: I->K,MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKF...
...,...,...,...,...,...
1833,L30M,1863,1,29: L->M,MDLSALRVEEVQNVINAMQKILECPICLEMIKEPVSTKCDHIFCKF...
1834,V14D,1863,1,13: V->D,MDLSALRVEEVQNDINAMQKILECPICLELIKEPVSTKCDHIFCKF...
1835,V14F,1863,1,13: V->F,MDLSALRVEEVQNFINAMQKILECPICLELIKEPVSTKCDHIFCKF...
1836,V14L,1863,1,13: V->L,MDLSALRVEEVQNLINAMQKILECPICLELIKEPVSTKCDHIFCKF...


## 2. Mutations & clinical labels

`sequence_metadata` is the richest per-variant table: it parses the mutation and attaches ClinVar. `annotations_dataframe` is the trimmed label table the downstream notebook uses for AUC (`annotation` is the benign/pathogenic call; `stars` is the ClinVar review-star confidence).

In [4]:
metadata = state["sequence_metadata"]
print(f"sequence_metadata: {metadata.shape}  (one row per variant)")
print("columns:", list(metadata.columns), "\n")
display(metadata.head(5))

sequence_metadata: (1837, 22)  (one row per variant)
columns: ['SequenceIndex', 'mutant', 'original_mutant', 'wt_aa', 'position', 'original_position', 'mutant_aa', 'n_mutation_sites', 'mutation_sites', 'is_synonymous', 'is_stop', 'sequence_length', 'original_sequence_length', 'analysis_sequence_start', 'analysis_sequence_end', 'analysis_sequence_transform', 'functional_score', 'has_clinvar', 'clinvar_significance_normalized', 'clinvar_review_status', 'clinvar_review_stars', 'annotation'] 



,SequenceIndex,mutant,original_mutant,wt_aa,position,original_position,mutant_aa,n_mutation_sites,mutation_sites,is_synonymous,is_stop,sequence_length,original_sequence_length,analysis_sequence_start,analysis_sequence_end,analysis_sequence_transform,functional_score,has_clinvar,clinvar_significance_normalized,clinvar_review_status,clinvar_review_stars,annotation
0,I1855M,I1855M,I1855M,I,1855,1855,M,1,[1854],False,False,1863,1863,1,1863,full,0.021941,True,other,no classification provided,0,NaN
1,I1855R,I1855R,I1855R,I,1855,1855,R,1,[1854],False,False,1863,1863,1,1863,full,-0.464328,True,other,no classification provided,0,NaN
2,I1855T,I1855T,I1855T,I,1855,1855,T,1,[1854],False,False,1863,1863,1,1863,full,-0.291519,True,other,"criteria provided, conflicting classifications",1,NaN
3,I1855K,I1855K,I1855K,I,1855,1855,K,1,[1854],False,False,1863,1863,1,1863,full,-0.863844,True,uncertain significance,"criteria provided, single submitter",1,NaN
4,I1855L,I1855L,I1855L,I,1855,1855,L,1,[1854],False,False,1863,1863,1,1863,full,0.007632,True,other,no classification provided,0,NaN


In [5]:
annotations = state["annotations_dataframe"]
print(f"annotations_dataframe: {annotations.shape}")
display(annotations.head(5))

print("\nannotation value counts (benign / pathogenic / unlabeled):")
display(annotations["annotation"].value_counts(dropna=False))
print("ClinVar review-star distribution:")
display(annotations["stars"].value_counts().sort_index())

annotations_dataframe: (1837, 9)


,SequenceIndex,mutant,annotation,has_clinvar,clinvar_significance_normalized,clinvar_review_status,clinvar_review_stars,functional_score,stars
0,I1855M,I1855M,NaN,True,other,no classification provided,0,0.021941,0
1,I1855R,I1855R,NaN,True,other,no classification provided,0,-0.464328,0
2,I1855T,I1855T,NaN,True,other,"criteria provided, conflicting classifications",1,-0.291519,1
3,I1855K,I1855K,NaN,True,uncertain significance,"criteria provided, single submitter",1,-0.863844,1
4,I1855L,I1855L,NaN,True,other,no classification provided,0,0.007632,0



annotation value counts (benign / pathogenic / unlabeled):


annotation
NaN           1534
pathogenic     196
benign         107
Name: count, dtype: int64

ClinVar review-star distribution:


stars
0    898
1    646
2    204
3     89
Name: count, dtype: int64

In [6]:
# Live ClinVar lookup for every nucleotide HGVS label mapped to the 1,838 protein sequences.
import os
import re
import shlex
import time
from datetime import datetime, timezone

import requests

CLINVAR_CACHE_PATH = TABLE_DIR / f"{DATASET}_live_clinvar_by_hgvs.csv"
FORCE_REFRESH = False       # True re-queries records already present in the checkpoint.
CHECKPOINT_EVERY = 25       # Save progress this often so an interrupted run can resume.


def _shell_export(name):
    """Read an exported credential without printing it; also works in kernels not launched from bash."""
    if value := os.environ.get(name):
        return value
    for rc_path in (Path.home() / ".bashrc", Path.home() / ".bash_profile"):
        if not rc_path.is_file():
            continue
        for line in rc_path.read_text(errors="ignore").splitlines():
            match = re.match(rf"^\s*export\s+{re.escape(name)}\s*=\s*(.*)$", line)
            if match:
                parsed = shlex.split(match.group(1), comments=True, posix=True)
                return parsed[0] if parsed else None
    return None


email = _shell_export("CLINVAR_EMAIL")
api_key = _shell_export("CLINVAR_TOKEN")
if not email or not api_key:
    raise RuntimeError("CLINVAR_EMAIL and CLINVAR_TOKEN must be exported or defined in ~/.bashrc")

eutils = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"
common_params = {"tool": "clinprotgym_brca1_overview", "email": email, "api_key": api_key}


def _clinvar_get(session, endpoint, params):
    """Issue a request while keeping the API key out of any displayed error."""
    try:
        response = session.get(f"{eutils}/{endpoint}", params=params, timeout=30)
        response.raise_for_status()
        return response.json()
    except (requests.RequestException, ValueError) as exc:
        raise RuntimeError(f"ClinVar request failed ({type(exc).__name__})") from None


REVIEW_STATUS_STARS = {
    "practice guideline": 4,
    "reviewed by expert panel": 3,
    "criteria provided, multiple submitters, no conflicts": 2,
    "criteria provided, single submitter": 1,
    "criteria provided, conflicting classifications": 1,
    "criteria provided, conflicting interpretations": 1,
}


def _clinvar_query_labels(hgvs_nt):
    """Try the supplied label first, then BRCA1 transcript-version aliases."""
    labels = [str(hgvs_nt)]
    match = re.fullmatch(r"NM_007294\.\d+:(c\..+)", str(hgvs_nt))
    if match:
        labels.extend([f"NM_007294.3:{match.group(1)}", f"NM_007294.4:{match.group(1)}"])
    return list(dict.fromkeys(labels))


def _lookup_clinvar_hgvs(session, hgvs_nt):
    for query_label in _clinvar_query_labels(hgvs_nt):
        search = _clinvar_get(session, "esearch.fcgi", {
            "db": "clinvar", "term": f'"{query_label}"', "retmode": "json",
            "retmax": 20, **common_params,
        })
        time.sleep(0.11)  # Stay below NCBI's API-key request limit.
        variation_ids = search.get("esearchresult", {}).get("idlist", [])
        if not variation_ids:
            continue

        summary = _clinvar_get(session, "esummary.fcgi", {
            "db": "clinvar", "id": ",".join(variation_ids), "retmode": "json",
            **common_params,
        })
        time.sleep(0.11)
        records = summary.get("result", {})
        c_expression = query_label.split(":", 1)[-1]
        for variation_id in variation_ids:
            record = records.get(variation_id, {})
            if c_expression not in str(record.get("title", "")):
                continue
            classification = record.get("germline_classification", {}) or {}
            review_status = classification.get("review_status") or ""
            return {
                "hgvs_nt": hgvs_nt,
                "hgvs_nt_query": query_label,
                "variation_id": variation_id,
                "accession": record.get("accession"),
                "title": record.get("title"),
                "classification": classification.get("description"),
                "review_status": review_status,
                "review_stars": REVIEW_STATUS_STARS.get(review_status.lower(), 0),
                "last_evaluated": classification.get("last_evaluated"),
                "clinvar_url": f"https://www.ncbi.nlm.nih.gov/clinvar/variation/{variation_id}/",
                "lookup_status": "found",
                "lookup_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            }
    return {
        "hgvs_nt": hgvs_nt,
        "lookup_status": "not_found",
        "lookup_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    }


def _write_checkpoint(cache_df):
    cache_df.drop_duplicates("hgvs_nt", keep="last").to_csv(CLINVAR_CACHE_PATH, index=False)


variant_map = pd.read_csv(
    TABLE_DIR / f"{DATASET}_hgvs_variant_map.csv",
    dtype={"hgvs_nt": str, "protein_sequence_index": str, "mutant": str},
)
query_df = (
    variant_map[["protein_sequence_index", "mutant", "mavedb_accession", "hgvs_nt"]]
    .dropna(subset=["hgvs_nt"])
    .drop_duplicates("hgvs_nt")
)

if CLINVAR_CACHE_PATH.is_file() and not FORCE_REFRESH:
    clinvar_cache = pd.read_csv(CLINVAR_CACHE_PATH, dtype={"hgvs_nt": str, "variation_id": str})
else:
    clinvar_cache = pd.DataFrame(columns=["hgvs_nt", "lookup_status"])

reusable = clinvar_cache[
    ~clinvar_cache["lookup_status"].fillna("").str.startswith("request_error")
]
completed_hgvs = set(reusable["hgvs_nt"].dropna())
missing = query_df[~query_df["hgvs_nt"].isin(completed_hgvs)]
print(f"HGVS records: {len(query_df):,}; cached: {len(completed_hgvs):,}; to query: {len(missing):,}")

fetched_rows = []
with requests.Session() as session:
    for n, hgvs_nt in enumerate(missing["hgvs_nt"], start=1):
        try:
            fetched_rows.append(_lookup_clinvar_hgvs(session, hgvs_nt))
        except RuntimeError as exc:
            fetched_rows.append({
                "hgvs_nt": hgvs_nt,
                "lookup_status": f"request_error: {exc}",
                "lookup_timestamp_utc": datetime.now(timezone.utc).isoformat(),
            })

        if n % CHECKPOINT_EVERY == 0:
            clinvar_cache = pd.concat([clinvar_cache, pd.DataFrame(fetched_rows)], ignore_index=True)
            clinvar_cache = clinvar_cache.drop_duplicates("hgvs_nt", keep="last")
            _write_checkpoint(clinvar_cache)
            fetched_rows = []
            print(f"queried {n:,}/{len(missing):,}; checkpoint: {CLINVAR_CACHE_PATH.name}")

if fetched_rows:
    clinvar_cache = pd.concat([clinvar_cache, pd.DataFrame(fetched_rows)], ignore_index=True)
clinvar_cache = clinvar_cache.drop_duplicates("hgvs_nt", keep="last")
for column in [
    "variation_id", "accession", "classification", "review_status",
    "review_stars", "lookup_status",
]:
    if column not in clinvar_cache:
        clinvar_cache[column] = pd.NA
_write_checkpoint(clinvar_cache)

# One row per nucleotide HGVS record (2,086 rows for this dataset).
clinvar_by_hgvs = query_df.merge(clinvar_cache, on="hgvs_nt", how="left")
clinvar_by_hgvs["review_stars"] = pd.to_numeric(clinvar_by_hgvs["review_stars"], errors="coerce")


def _join_unique(values):
    return " | ".join(dict.fromkeys(str(value) for value in values if pd.notna(value) and str(value)))


def _normalize_classification(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).lower()
    if "conflict" in value:
        return "other"
    if "pathogenic" in value:
        return "pathogenic"
    if "benign" in value:
        return "benign"
    if "uncertain" in value:
        return "uncertain significance"
    return pd.NA


clinvar_by_hgvs["fresh_annotation"] = clinvar_by_hgvs["classification"].map(_normalize_classification)


# Aggregate synonymous nucleotide encodings onto the 1,837 mutant proteins, then add wild type.
protein_clinvar = (
    clinvar_by_hgvs.groupby("mutant", as_index=False)
    .agg(
        fresh_n_hgvs_nt=("hgvs_nt", "nunique"),
        fresh_n_clinvar_records=("variation_id", lambda values: values.notna().sum()),
        fresh_hgvs_nt=("hgvs_nt", _join_unique),
        fresh_clinvar_accessions=("accession", _join_unique),
        fresh_annotations=("fresh_annotation", _join_unique),
        fresh_classifications=("classification", _join_unique),
        fresh_review_statuses=("review_status", _join_unique),
        fresh_max_review_stars=("review_stars", "max"),
        fresh_lookup_statuses=("lookup_status", _join_unique),
    )
    .rename(columns={"mutant": "SequenceIndex"})
)
default_clinvar = annotations[["SequenceIndex", "annotation", "stars"]].rename(columns={
    "annotation": "default_annotation",
    "stars": "default_review_stars",
})
protein_index = pd.DataFrame({"SequenceIndex": ["__wildtype__", *metadata["SequenceIndex"].astype(str)]})
current_clinvar_annotations = (
    protein_index
    .merge(default_clinvar, on="SequenceIndex", how="left")
    .merge(protein_clinvar, on="SequenceIndex", how="left")
)
current_clinvar_annotations["annotation_matches_any_fresh"] = current_clinvar_annotations.apply(
    lambda row: (
        pd.NA
        if pd.isna(row["default_annotation"]) or pd.isna(row["fresh_annotations"]) or not row["fresh_annotations"]
        else str(row["default_annotation"]) in str(row["fresh_annotations"]).split(" | ")
    ),
    axis=1,
)
current_clinvar_annotations.loc[
    current_clinvar_annotations["SequenceIndex"].eq("__wildtype__"), "fresh_lookup_statuses"
] = "not_applicable_wildtype"

assert len(current_clinvar_annotations) == len(seq_map) == 1838
print(f"ClinVar checkpoint: {CLINVAR_CACHE_PATH}")
print(f"Per-HGVS table: {clinvar_by_hgvs.shape}; per-protein table: {current_clinvar_annotations.shape}")
display(current_clinvar_annotations)

HGVS records: 2,086; cached: 675; to query: 1,411
queried 25/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 50/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 75/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 100/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 125/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 150/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 175/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 200/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 225/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 250/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 275/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 300/1,411; checkpoint: MV_BRCA1_Findlay_2018_live_clinvar_by_hgvs.csv
queried 325/1,411

,SequenceIndex,default_annotation,default_review_stars,fresh_n_hgvs_nt,fresh_n_clinvar_records,fresh_hgvs_nt,fresh_clinvar_accessions,fresh_annotations,fresh_classifications,fresh_review_statuses,fresh_max_review_stars,fresh_lookup_statuses,annotation_matches_any_fresh
0,__wildtype__,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,not_applicable_wildtype,<NA>
1,I1855M,NaN,0.0,1.0,1.0,NM_007294.3:c.5565A>G,VCV000869059,,,,0.0,found,<NA>
2,I1855R,NaN,0.0,1.0,1.0,NM_007294.3:c.5564T>G,VCV000869058,,,,0.0,found,<NA>
3,I1855T,NaN,1.0,1.0,1.0,NM_007294.3:c.5564T>C,VCV000482917,other,Conflicting classifications of pathogenicity,"criteria provided, conflicting classifications",1.0,found,<NA>
4,I1855K,NaN,1.0,1.0,1.0,NM_007294.3:c.5564T>A,VCV000868679,uncertain significance,Uncertain significance,"criteria provided, single submitter",1.0,found,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1833,L30M,NaN,0.0,1.0,1.0,NM_007294.3:c.88T>A,VCV000865099,,,,0.0,found,<NA>
1834,V14D,NaN,0.0,1.0,1.0,NM_007294.3:c.41T>A,VCV000865031,,,,0.0,found,<NA>
1835,V14F,NaN,2.0,1.0,1.0,NM_007294.3:c.40G>T,VCV000848933,uncertain significance,Uncertain significance,"criteria provided, multiple submitters, no con...",2.0,found,<NA>
1836,V14L,NaN,0.0,1.0,1.0,NM_007294.3:c.40G>C,VCV000869093,,,,0.0,found,<NA>


In [20]:
old_annotations = current_clinvar_annotations["default_annotation"].to_list()

fresh_annotations = current_clinvar_annotations["fresh_annotations"].to_list()

In [21]:
# print the number of pathogenic, benign for both the default and fresh annotations, as well as the number of variants where they match or don't match.
default_counts = current_clinvar_annotations["default_annotation"].value_counts(dropna=False)
fresh_counts = current_clinvar_annotations["fresh_annotations"].value_counts(dropna=False)

In [22]:
default_counts

default_annotation
NaN           1535
pathogenic     196
benign         107
Name: count, dtype: int64

In [23]:
fresh_counts

fresh_annotations
                                       882
uncertain significance                 351
other                                  289
pathogenic                             195
benign                                  97
uncertain significance | other           8
benign | other                           6
other | uncertain significance           3
other | benign                           2
uncertain significance | benign          2
NaN                                      1
benign | uncertain significance          1
pathogenic | uncertain significance      1
Name: count, dtype: int64

In [7]:
# mutation-site map: mutant -> list of 0-based residue positions that changed
site_map = state["sequence_to_mutation_sites"]
site_df = pd.DataFrame(
    [(m, sites) for m, sites in list(site_map.items())[:6]],
    columns=["mutant", "mutation_sites (0-based)"],
)
display(site_df)

,mutant,mutation_sites (0-based)
0,__wildtype__,[]
1,I1855M,[1854]
2,I1855R,[1854]
3,I1855T,[1854]
4,I1855K,[1854]
5,I1855L,[1854]


## 3. Count data

Three views of the experimental measurements:

* **`adapter_counts.csv`** — the wide raw table: one row per variant, one column per sequencing condition (`count__count_day5_rep1`, `day11_rep2`, `library`, `negative_control`, `rna_rep*`, ...), plus the published `functional_score` and ClinVar columns.
* **`sequence_dataframe`** — the same counts reshaped **long** for popDMS: `(SequenceIndex, Replicate, Generation, Frequency, CountColumn)`. This is the trajectory input to the selection-coefficient inference.
* **`scores_dataframe`** — just the scalar functional score per variant.

In [8]:
counts = pd.read_csv(SEQ_DIR / f"{DATASET}_adapter_counts.csv")
count_cols = [c for c in counts.columns if c.startswith("count__")]
print(f"adapter_counts.csv: {counts.shape}")
print("raw count columns:", count_cols, "\n")
# show identity + counts + score, but drop the giant mutated_sequence column for readability
display(counts[["mutant", "SequenceIndex", *count_cols, "functional_score"]].head(6))

adapter_counts.csv: (1837, 28)
raw count columns: ['count__count_day11_rep1', 'count__count_day11_rep2', 'count__count_day5_rep1', 'count__count_day5_rep2', 'count__count_library', 'count__count_negative_control', 'count__count_rna_rep1', 'count__count_rna_rep2'] 



,mutant,SequenceIndex,count__count_day11_rep1,count__count_day11_rep2,count__count_day5_rep1,count__count_day5_rep2,count__count_library,count__count_negative_control,count__count_rna_rep1,count__count_rna_rep2,functional_score
0,I1855M,I1855M,3037,2334,2908,2760,436,2,1406.0,666.0,0.021941
1,I1855R,I1855R,2701,1530,3132,3878,505,2,2449.0,1310.0,-0.464328
2,I1855T,I1855T,2124,2750,2905,4174,518,1,1146.0,3101.0,-0.291519
3,I1855K,I1855K,1016,1281,2372,2841,388,1,1426.0,350.0,-0.863844
4,I1855L,I1855L,6540,4835,6870,8126,950,4,2782.0,3759.0,0.007632
5,I1855V,I1855V,3091,2166,2559,3343,432,79,973.0,566.0,0.046278


In [9]:
seq_df = state["sequence_dataframe"]
print(f"sequence_dataframe (long popDMS trajectory table): {seq_df.shape}")
display(seq_df.head(8))

print("\nCountColumn values (each is one experimental condition/timepoint):")
display(seq_df["CountColumn"].value_counts())

sequence_dataframe (long popDMS trajectory table): (11022, 7)


,SequenceIndex,mutant,Replicate,ReplicateName,Generation,Frequency,CountColumn
0,I1855M,I1855M,1,rep1,0.0,436.0,count__count_library
1,I1855R,I1855R,1,rep1,0.0,505.0,count__count_library
2,I1855T,I1855T,1,rep1,0.0,518.0,count__count_library
3,I1855K,I1855K,1,rep1,0.0,388.0,count__count_library
4,I1855L,I1855L,1,rep1,0.0,950.0,count__count_library
5,I1855V,I1855V,1,rep1,0.0,432.0,count__count_library
6,L1854R,L1854R,1,rep1,0.0,406.0,count__count_library
7,L1854P,L1854P,1,rep1,0.0,371.0,count__count_library



CountColumn values (each is one experimental condition/timepoint):


CountColumn
count__count_library       3674
count__count_day5_rep1     1837
count__count_day11_rep1    1837
count__count_day5_rep2     1837
count__count_day11_rep2    1837
Name: count, dtype: int64

In [10]:
scores = state["scores_dataframe"]
print(f"scores_dataframe: {scores.shape}")
display(scores.head(5))
print(scores[["functional_score", "score"]].describe())

scores_dataframe: (1837, 4)


,SequenceIndex,mutant,functional_score,score
0,I1855M,I1855M,0.021941,0.021941
1,I1855R,I1855R,-0.464328,-0.464328
2,I1855T,I1855T,-0.291519,-0.291519
3,I1855K,I1855K,-0.863844,-0.863844
4,I1855L,I1855L,0.007632,0.007632


       functional_score        score
count       1837.000000  1837.000000
mean          -0.601125    -0.601125
std            0.890618     0.890618
min           -3.718221    -3.718221
25%           -1.029762    -1.029762
50%           -0.261441    -0.261441
75%            0.025227     0.025227
max            1.094138     1.094138


## 4. Protein embeddings

Embeddings are **not** in the state pickle — they are separate per-model, per-layer pickle files in `sequence_data/`. Two ESM-C models are cached (each with `max_pool`, `mean_pool`, and `per_residue` variants; this notebook uses `max_pool`):

* `biohub__ESMC-300M` &rarr; 960-dim vectors (max_pool layers 0&ndash;30)
* `biohub__ESMC-600M` &rarr; 1152-dim vectors (max_pool layers 0&ndash;36)

Each layer has two files:
* `*_Layer_{L}_seq_to_features.pkl` &mdash; `dict[mutant -> np.ndarray]`, the max-pooled embedding for that variant.
* `*_Layer_{L}_none_inference_results.pkl` &mdash; a `popDMS.InferenceResult` with selection coefficients (`s`, `s_joint`, error bars) fit from the embedding trajectory.

In [11]:
# Inventory: which models/layers have cached max-pooled embeddings?
# (mean_pool / per_residue variants also exist on disk; we focus on max_pool here.)
feat_files = sorted(glob.glob(str(SEQ_DIR / "*_max_pool_none_Layer_*_seq_to_features.pkl")))
inv = {}
for f in feat_files:
    name = Path(f).name
    model = name.split("_max_pool")[0].replace(f"{DATASET}_", "")
    layer = int(name.split("Layer_")[1].split("_")[0])
    inv.setdefault(model, []).append(layer)
for model, layers in sorted(inv.items()):
    print(f"{model}: {len(layers)} layers  (layers {min(layers)}..{max(layers)})")

biohub__ESMC-300M: 31 layers  (layers 0..30)
biohub__ESMC-600M: 37 layers  (layers 0..36)


In [12]:
# Load one embedding layer and expose it as a (variant x dim) DataFrame.
EMB_MODEL = "biohub__ESMC-300M"
EMB_LAYER = 0
emb_path = SEQ_DIR / f"{DATASET}_{EMB_MODEL}_max_pool_none_Layer_{EMB_LAYER}_seq_to_features.pkl"
with emb_path.open("rb") as handle:
    emb = pickle.load(handle)  # dict: mutant -> np.ndarray(dim,)

dim = next(iter(emb.values())).shape[0]
print(f"{EMB_MODEL} Layer {EMB_LAYER}: {len(emb)} variants x {dim} dims")

emb_df = pd.DataFrame.from_dict(emb, orient="index")
emb_df.columns = [f"dim_{i}" for i in range(dim)]
emb_df.index.name = "SequenceIndex"
display(emb_df.iloc[:5, :8])  # first 5 variants, first 8 dims

biohub__ESMC-300M Layer 0: 1838 variants x 960 dims


,dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7
SequenceIndex,,,,,,,,
A1641D,1.543199,1.351962,2.065732,0.206562,1.683196,0.908416,1.167336,1.097027
A1641G,1.543199,1.351962,2.065732,0.206562,1.683196,0.908416,1.167336,1.097027
A1641P,1.543199,1.351962,2.065732,0.206562,1.683196,0.908416,1.167336,1.097027
A1641S,1.543199,1.351962,2.065732,0.206562,1.683196,0.908416,1.167336,1.097027
A1641T,1.543199,1.351962,2.065732,0.206562,1.683196,0.908416,1.167336,1.097027


In [13]:
# popDMS InferenceResult that accompanies each embedding layer.
inf_path = SEQ_DIR / f"{DATASET}_{EMB_MODEL}_max_pool_none_Layer_{EMB_LAYER}_none_inference_results.pkl"
with inf_path.open("rb") as handle:
    inference = pickle.load(handle)

print(f"InferenceResult type: {type(inference)}")
for attr, value in vars(inference).items():
    shape = getattr(value, "shape", None)
    detail = f"shape={shape}" if shape is not None else f"{type(value).__name__} len={len(value) if hasattr(value, '__len__') else '-'}"
    print(f"  {attr:20s} {detail}")

InferenceResult type: <class 'popDMS.InferenceResult'>
  dx                   list len=2
  icov                 list len=2
  s                    shape=(2, 960)
  s_joint              shape=(960,)
  gamma_opt            shape=()
  x_array              list len=2
  error_bars           shape=(2, 960)
  s_joint_error_bars   shape=(960,)


### 4b. SAE features and LLR baseline (optional / downstream artifacts)

These are produced by the pipeline jobs. SAE feature pickles have the same `dict[mutant -> np.ndarray]` shape as the raw embeddings, but the vectors are the learned **sparse** features (dimensionality is the SAE width, e.g. 12800 for the `nf12800` run — most entries are zero). The LLR baseline is a simple per-variant fitness CSV.

In [14]:
# One SAE feature file, discovered from a completed task_result.json (if present).
sae_results = sorted(glob.glob(str(DATASET_ROOT / "jobs" / "fixed_sae_model_layer_array" / "*" / "Layer_*" / "task_result.json")))
sae_feat = None
for tr_path in sae_results:
    import json
    tr = json.loads(Path(tr_path).read_text())
    fp = tr.get("feature_path", "")
    if tr.get("status") == "ok" and fp and Path(fp).is_file():
        sae_feat = fp
        break

if sae_feat is None:
    print("No completed SAE feature file found (run the SAE jobs first).")
else:
    with open(sae_feat, "rb") as handle:
        sae = pickle.load(handle)
    sdim = next(iter(sae.values())).shape[0]
    print(f"SAE features: {len(sae)} variants x {sdim} learned features")
    print(f"source: {Path(sae_feat).name}")
    sae_df = pd.DataFrame.from_dict(sae, orient="index")
    sae_df.index.name = "SequenceIndex"
    # SAE features are sparse; show how many are non-zero per variant
    display(pd.DataFrame({"n_active_features": (sae_df != 0).sum(axis=1)}).head(5))

SAE features: 1837 variants x 12800 learned features
source: MV_BRCA1_Findlay_2018_biohub__ESMC-300M_max_pool_DeltaEmbSAE_Layer_0_seq_to_features.pkl


,n_active_features
SequenceIndex,
A1641D,0
A1641G,0
A1641P,0
A1641S,0
A1641T,0


In [15]:
# LLR (log-likelihood-ratio) baseline fitness table, if generated.
llr_files = sorted(glob.glob(str(TABLE_DIR / f"{DATASET}_*_llr_fitness.csv")))
if not llr_files:
    print("No LLR fitness tables found yet.")
else:
    for f in llr_files:
        print(Path(f).name)
    llr = pd.read_csv(llr_files[0])
    print(f"\n{Path(llr_files[0]).name}: {llr.shape}")
    display(llr.head(5))

MV_BRCA1_Findlay_2018_biohub__ESMC-300M_llr_fitness.csv
MV_BRCA1_Findlay_2018_biohub__ESMC-600M_llr_fitness.csv

MV_BRCA1_Findlay_2018_biohub__ESMC-300M_llr_fitness.csv: (1837, 3)


,SequenceIndex,fitness,mutant
0,I1855M,-2.926415,I1855M
1,I1855R,-6.324734,I1855R
2,I1855T,-4.171642,I1855T
3,I1855K,-6.449422,I1855K
4,I1855L,-1.376818,I1855L


## 5. One joined view

All tables key on `SequenceIndex`. Here they are stitched into a single per-variant DataFrame: identity + functional score + clinical label + one embedding dimension, to show how the layers line up.

In [16]:
joined = (
    metadata[["SequenceIndex", "wt_aa", "position", "mutant_aa", "is_synonymous", "is_stop", "functional_score"]]
    .merge(annotations[["SequenceIndex", "annotation", "stars"]], on="SequenceIndex", how="left")
)
joined = joined.merge(
    emb_df[["dim_0"]].rename(columns={"dim_0": f"{EMB_MODEL}_L{EMB_LAYER}_dim0"}),
    left_on="SequenceIndex", right_index=True, how="left",
)
print(f"joined per-variant table: {joined.shape}")
display(joined.head(10))

joined per-variant table: (1837, 10)


,SequenceIndex,wt_aa,position,mutant_aa,is_synonymous,is_stop,functional_score,annotation,stars,biohub__ESMC-300M_L0_dim0
0,I1855M,I,1855,M,False,False,0.021941,NaN,0,1.543199
1,I1855R,I,1855,R,False,False,-0.464328,NaN,0,1.543199
2,I1855T,I,1855,T,False,False,-0.291519,NaN,1,1.543199
3,I1855K,I,1855,K,False,False,-0.863844,NaN,1,1.543199
4,I1855L,I,1855,L,False,False,0.007632,NaN,0,1.543199
5,I1855V,I,1855,V,False,False,0.046278,NaN,2,1.543199
6,L1854R,L,1854,R,False,False,0.312452,NaN,0,1.543199
7,L1854P,L,1854,P,False,False,-1.343872,pathogenic,3,1.543199
8,L1854Q,L,1854,Q,False,False,0.259387,NaN,0,1.543199
9,L1854V,L,1854,V,False,False,0.158476,NaN,0,1.543199


In [17]:
rows_df = pd.DataFrame(rows)

In [18]:
prot_seqs = rows_df["protein_sequence"].to_list()

In [19]:
len(prot_seqs), len(set(prot_seqs)), prot_seqs[:5]

(1838,
 1838,
 ['MDLSALRVEEVQNVINAMQKILECPICLELIKEPVSTKCDHIFCKFCMLKLLNQKKGPSQCPLCKNDITKRSLQESTRFSQLVEELLKIICAFQLDTGLEYANSYNFAKKENNSPEHLKDEVSIIQSMGYRNRAKRLLQSEPENPSLQETSLSVQLSNLGTVRTLRTKQRIQPQKTSVYIELGSDSSEDTVNKATYCSVGDQELLQITPQGTRDEISLDSAKKAACEFSETDVTNTEHHQPSNNDLNTTEKRAAERHPEKYQGSSVSNLHVEPCGTNTHASSLQHENSSLLLTKDRMNVEKAEFCNKSKQPGLARSQHNRWAGSKETCNDRRTPSTEKKVDLNADPLCERKEWNKQKLPCSENPRDTEDVPWITLNSSIQKVNEWFSRSDELLGSDDSHDGESESNAKVADVLDVLNEVDEYSGSSEKIDLLASDPHEALICKSERVHSKSVESNIEDKIFGKTYRKKASLPNLSHVTENLIIGAFVTEPQIIQERPLTNKLKRKRRPTSGLHPEDFIKKADLAVQKTPEMINQGTNQTEQNGQVMNITNSGHENKTKGDSIQNEKNPNPIESLEKESAFKTKAEPISSSISNMELELNIHNSKAPKKNRLRRKSSTRHIHALELVVSRNLSPPNCTELQIDSCSSSEEIKKKKYNQMPVRHSRNLQLMEGKEPATGAKKSNKPNEQTSKRHDSDTFPELKLTNAPGSFTKCSNTSELKEFVNPSLPREEKEEKLETVKVSNNAEDPKDLMLSGERVLQTERSVESSSISLVPGTDYGTQESISLLEVSTLGKAKTEPNKCVSQCAAFENPKGLIHGCSKDNRNDTEGFKYPLGHEVNHSRETSIEMEESELDAQYLQNTFKVSKRQSFAPFSNPGNAEEECATFSAHSGSLKKQSPKVTFECEQKEENQGKNESNIKPVQTVNITAGFPVVGQKDKPVDNAKCSIKGGSRFCLSSQFRGNETGLITPNKHGLLQNPYRIPPL